# Wildfire Intensity Prediction: Data Engineering Pipeline

This notebook integrates five Canadian wildfire and environmental data sources: NASA FIRMS, NASA POWER, CWFIS, MRDEM, and FBP.

**Final dataset:** 'data/wildfire_ml_dataset.parquet'

- The dataset contains 203,744 observations and 16 columns.
- Some steps require large files, remote raster access, or thousands of API requests.

In [1]:
# Common imports
import numpy as np
import os
import tarfile
from six.moves import urllib
import pandas as pd

from pathlib import Path
import requests
import rasterio
from rasterio.warp import transform
from rasterio.windows import Window

# To make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

# Ignore useless warnings
import warnings
warnings.filterwarnings(action="ignore", message="^internal gelsd")

## 1. NASA FIRMS: Historical Fire Data

### 1.1. Identify Source Files

Locate the downloaded VIIRS S-NPP fire detection files for Canada (2014–2026).

In [2]:
# Locate NASA FIRMS files
firms_dir = Path("data/DL_FIRE_SV-C2_852451")
firms_files = sorted(firms_dir.rglob("*.csv"))

print("FIRMS CSV files:", len(firms_files))

for file in firms_files:
    print(file.name, "-", round(file.stat().st_size / 1024**2, 2), "MB")

FIRMS CSV files: 2
fire_archive_SV-C2_852451.csv - 535.11 MB
fire_nrt_SV-C2_852451.csv - 7.84 MB


### 1.2. Load and Inspect FIRMS Data

Inspect the structure, columns, and sample records of the historical VIIRS S-NPP dataset.

In [3]:
# Inspect FIRMS data without loading the full dataset
if not firms_files:
    raise FileNotFoundError("No FIRMS CSV files found.")

firms_sample = pd.read_csv(firms_files[0], nrows=5)

print("Number of source files:", len(firms_files))
print("Columns:", firms_sample.columns.tolist())

display(firms_sample)

Number of source files: 2
Columns: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,42.81144,-80.09736,313.46,0.49,0.65,2012-01-20,618,SNPP,SNPP,n,2,259.67,3.25,N,2
1,42.80534,-80.09772,331.67,0.49,0.65,2012-01-20,618,SNPP,SNPP,n,2,262.74,3.25,N,2
2,46.57152,-80.79647,333.81,0.43,0.46,2012-01-20,758,SNPP,SNPP,n,2,242.10,5.75,N,2
3,43.26311,-79.80861,314.54,0.58,0.52,2012-01-20,758,SNPP,SNPP,n,2,263.69,2.52,N,2
4,42.80490,-80.09639,315.34,0.57,0.52,2012-01-20,759,SNPP,SNPP,n,2,262.05,3.19,N,2


### 1.3. Historical Data Coverage

Verify the date range and number of fire detections in each source file.

In [4]:
# Load processed FIRMS data when available
firms_path = Path("data/firms_daily.parquet")
USE_CACHED_FIRMS = firms_path.exists()

if USE_CACHED_FIRMS:
    firms_daily = pd.read_parquet(firms_path)
    print("Processed FIRMS data loaded:", firms_daily.shape)
else:
    firms = pd.concat(
        [pd.read_csv(file) for file in firms_files],
        ignore_index=True
    )
    firms["acq_date"] = pd.to_datetime(firms["acq_date"])

    print("Raw FIRMS data:", firms.shape)
    print("Date range:", firms["acq_date"].min(),
          "to", firms["acq_date"].max())

Processed FIRMS data loaded: (203744, 7)


### 1.4. Filter Study Period

Keep NASA FIRMS observations from January 2014 onward.

The study starts in 2014 to align with the historical FBP Fuel Types 2014b dataset.

In [5]:
# Filter the study period only when needed
if not USE_CACHED_FIRMS:
    firms = firms[firms["acq_date"] >= "2014-01-01"].copy()

    print("Observations:", len(firms))
    print("Date range:", firms["acq_date"].min(),
          "to", firms["acq_date"].max())
else:
    print("FIRMS filtering skipped: processed data available.")

FIRMS filtering skipped: processed data available.


### 1.5. Fire Detection Types

Examine the different detection types in NASA FIRMS to identify vegetation fires and exclude other heat sources.

In [6]:
# Examine fire detection types only when needed
if not USE_CACHED_FIRMS:
    print(firms["type"].value_counts().sort_index())
else:
    print("FIRMS type filtering already completed.")

FIRMS type filtering already completed.


### 1.6. Filter Vegetation Fires

Keep only detections classified as presumed vegetation fires ('type = 0').

Exclude static heat sources ('type = 2') and offshore detections ('type = 3').

Source: [NASA FIRMS  VIIRS Data Description](https://firms.modaps.eosdis.nasa.gov/content/descriptions/FIRMS_VIIRS_Firehotspots.html)


In [7]:
# Keep presumed vegetation fires
if not USE_CACHED_FIRMS:
    firms = firms[firms["type"] == 0].copy()

    print("Vegetation fire observations:", len(firms))
else:
    print("Vegetation fire filtering skipped: processed data available.")

Vegetation fire filtering skipped: processed data available.


### 1.7. Data Quality Check

Check missing values, duplicate records, invalid coordinates, and negative FRP values before further processing.


In [8]:
# Check raw FIRMS data quality only when needed
if not USE_CACHED_FIRMS:
    print("Missing values:")
    print(firms.isna().sum())

    print("\nDuplicate rows:", firms.duplicated().sum())

    print("\nInvalid coordinates:",
          (~firms["latitude"].between(-90, 90) |
           ~firms["longitude"].between(-180, 180)).sum())

    print("\nNegative FRP values:", (firms["frp"] < 0).sum())
    print("Zero FRP values:", (firms["frp"] == 0).sum())
else:
    print("Raw FIRMS quality checks skipped: processed data available.")

Raw FIRMS quality checks skipped: processed data available.


### 1.8. Spatial and Daily Aggregation

Divide Canada into 0.5° × 0.5° grid cells to aggregate fire detections by location and date.

This facilitates integration with NASA POWER weather data.

**Decision:** Use a 0.5° grid as the initial spatial resolution. This is an approximate alignment with weather data and may be refined later.

In [9]:
# Create spatial grid coordinates only when needed
grid_size = 0.5

if not USE_CACHED_FIRMS:
    firms["grid_lat"] = np.floor(firms["latitude"] / grid_size) * grid_size
    firms["grid_lon"] = np.floor(firms["longitude"] / grid_size) * grid_size

    display(firms[["latitude", "longitude", "grid_lat", "grid_lon"]].head())
else:
    print("Spatial grid already available in processed FIRMS data.")

Spatial grid already available in processed FIRMS data.


In [10]:
# Aggregate fire detections only when needed
if not USE_CACHED_FIRMS:
    firms_daily = firms.groupby(
        ["grid_lat", "grid_lon", "acq_date"]
    ).agg(
        fire_count=("frp", "size"),
        max_frp=("frp", "max"),
        mean_frp=("frp", "mean"),
        total_frp=("frp", "sum")
    ).reset_index()

print("Daily observations:", firms_daily.shape)
display(firms_daily.head())

Daily observations: (203744, 7)


,grid_lat,grid_lon,acq_date,fire_count,max_frp,mean_frp,total_frp
0,41.5,-83.0,2019-04-06,1,3.19,3.190000,3.19
1,41.5,-83.0,2019-06-22,3,18.05,10.963333,32.89
2,41.5,-83.0,2019-12-17,1,12.75,12.750000,12.75
3,41.5,-83.0,2021-03-14,2,0.63,0.600000,1.20
4,41.5,-83.0,2024-02-07,2,4.67,4.220000,8.44


### 1.9. Validate Aggregated Data

Verify that each grid cell and date combination is unique and that the prediction target ('max_frp') contains no missing values.


In [11]:
# Validate daily aggregation
print("Duplicate grid-date records:",
      firms_daily.duplicated(["grid_lat", "grid_lon", "acq_date"]).sum())

print("Missing max_frp:", firms_daily["max_frp"].isna().sum())

print("Unique grid cells:",
      firms_daily[["grid_lat", "grid_lon"]].drop_duplicates().shape[0])

Duplicate grid-date records: 0
Missing max_frp: 0
Unique grid cells: 3977


### 1.10. FIRMS: Findings & Decisions

**Findings**

- Loaded historical NASA FIRMS VIIRS S-NPP observations for Canada.
- Selected the study period: 2014–2026.
- Retained 6,027,345 presumed vegetation fire detections (`type = 0`).
- No missing values, duplicate rows, invalid coordinates, or negative FRP values were found.
- Aggregated detections into 0.5° × 0.5° grid cells by day.
- Obtained 203,744 daily observations across 3,977 unique grid cells.
- No duplicate grid-date records or missing target values were found.

**Meaning of the Aggregated Fields**

| Field | Meaning |
|---|---|
| 'grid_lat' | Southern boundary of the grid cell |
| 'grid_lon' | Western boundary of the grid cell |
| 'acq_date' | Satellite acquisition date |
| 'fire_count' | Number of satellite fire detections |
| 'max_frp' | Maximum daily Fire Radiative Power (MW) |
| 'mean_frp' | Mean FRP of daily detections (MW) |
| 'total_frp' | Sum of FRP values from daily detections (MW) |

**Decisions**

- Use 'max_frp' as the ML prediction target.
- Retain the 0.5° spatial grid for the initial pipeline.
- Use the aggregated dataset for integration with weather, fire-weather indices, terrain, and fuel type.

**Limitations**

- Only grid-cell days with satellite detections are represented.
- A grid cell may contain multiple distinct fires.
- The grid does not exactly match the spatial resolution of the other datasets.


### 1.11. Save Processed FIRMS Data

Save the aggregated daily fire observations in Parquet format for efficient storage and reuse in the data integration pipeline.


In [12]:
# Save processed FIRMS data only when needed
output_path = Path("data/firms_daily.parquet")

if not USE_CACHED_FIRMS:
    firms_daily.to_parquet(output_path, index=False)
    print("FIRMS data saved.")
else:
    print("Existing FIRMS file reused.")

print("Observations:", len(firms_daily))
print("File size:", round(output_path.stat().st_size / 1024**2, 2), "MB")

Existing FIRMS file reused.
Observations: 203744
File size: 2.62 MB


## 2. NASA POWER: Historical Weather Data

### 2.1. Prepare Weather Locations

Identify the geographic locations of the FIRMS observations.

Weather data will be collected for Canada (2014-2026) using the NASA POWER Regional API.


In [13]:
# Prepare weather locations
grid_cells = firms_daily[["grid_lat", "grid_lon"]].drop_duplicates().copy()

grid_cells["latitude"] = grid_cells["grid_lat"] + 0.25
grid_cells["longitude"] = grid_cells["grid_lon"] + 0.25

print("Number of locations:", len(grid_cells))
display(grid_cells.head())

Number of locations: 3977


,grid_lat,grid_lon,latitude,longitude
0,41.5,-83.0,41.75,-82.75
6,42.0,-83.5,42.25,-83.25
18,42.0,-83.0,42.25,-82.75
155,42.0,-82.5,42.25,-82.25
366,42.0,-82.0,42.25,-81.75


### 2.2. Prepare Regional Download Areas

Group FIRMS locations into 8° × 8° regions to reduce the number of NASA POWER API requests.

The regional grid is used only for downloading weather data. The original 0.5° FIRMS grid remains unchanged.


In [14]:
# Create regional download areas
grid_cells["region_lat"] = np.floor(grid_cells["latitude"] / 8) * 8
grid_cells["region_lon"] = np.floor(grid_cells["longitude"] / 8) * 8

regions = grid_cells[["region_lat", "region_lon"]].drop_duplicates()

print("Number of regions:", len(regions))
display(regions.head())

Number of regions: 48


,region_lat,region_lon
0,40.0,-88.0
2118,40.0,-80.0
3255,40.0,-72.0
5458,40.0,-64.0
11378,40.0,-56.0


### 2.3. Configure Historical Weather Collection

Prepare NASA POWER Regional API requests for the 48 geographic regions covering FIRMS locations.

Daily weather data will be collected year by year from 2014 to 2026, using four meteorological variables.


In [15]:
# Weather collection configuration
weather_variables = ["T2M", "RH2M", "WS10M", "PRECTOTCORR"]
years = range(2014, 2027)

power_dir = Path("data/power")
power_dir.mkdir(parents=True, exist_ok=True)

print("Regions:", len(regions))
print("Variables:", len(weather_variables))
print("Years:", len(years))
print("Maximum requests:", len(regions) * len(weather_variables) * len(years))

Regions: 48
Variables: 4
Years: 13
Maximum requests: 2496


### 2.4. Download Historical Weather Data

Download NASA POWER daily weather data for Canada from 2014 to 2026.

Use 8° × 8° geographic regions and annual requests to respect NASA POWER API limits.

Save each regional dataset locally to support reproducibility and interrupted-download recovery.


In [16]:
# Download historical NASA POWER data
import time
RUN_POWER_DOWNLOAD = False

api_url = "https://power.larc.nasa.gov/api/temporal/daily/regional"
failed = []

for _, region in (regions.iterrows() if RUN_POWER_DOWNLOAD else []):
    lat, lon = region["region_lat"], region["region_lon"]

    for year in years:
        for variable in weather_variables:
            file = power_dir / f"POWER_{int(lat)}_{int(lon)}_{year}_{variable}.csv"

            if file.exists():
                continue

            params = {
                "latitude-min": lat,
                "latitude-max": lat + 8,
                "longitude-min": lon,
                "longitude-max": lon + 8,
                "parameters": variable,
                "community": "AG",
                "start": f"{year}0101",
                "end": "20261102" if year == 2026 else f"{year}1231",
                "time-standard": "UTC",
                "format": "CSV"
            }

            try:
                response = requests.get(api_url, params=params, timeout=180)
                response.raise_for_status()

                if not response.text.startswith("-BEGIN HEADER-"):
                    raise ValueError("Invalid CSV response")

                file.write_bytes(response.content)

            except Exception as e:
                failed.append(file.name)
                print("Failed:", file.name, str(e)[:80])

            time.sleep(1)

    print("Completed region:", lat, lon)

print("Downloaded files:", len(list(power_dir.glob("POWER_*.csv"))))
print("Failed requests:", len(failed))

Downloaded files: 2496
Failed requests: 0


### NASA POWER: Challenges

**Challenge:** Collecting 13 years of weather data across Canada requires 2,496 API requests, making the process very time-consuming.

**Solution:** Automated downloads, saved files locally, and enabled resuming without downloading existing files again.

**Limitation:** Failed or incomplete downloads must be checked before data integration.


### 2.5. Validate Downloaded Weather Files

Verify the downloaded NASA POWER files before combining the historical weather data.

In [17]:
# Validate downloaded files
power_files = list(power_dir.glob("POWER_*.csv"))

print("Number of files:", len(power_files))
print("Empty files:", sum(file.stat().st_size == 0 for file in power_files))

# Inspect one file
power_sample = pd.read_csv(power_files[0], skiprows=9, na_values=-999)

print("Sample shape:", power_sample.shape)
display(power_sample.head())

Number of files: 2496
Empty files: 0
Sample shape: (80665, 5)


,LAT,LON,YEAR,DOY,PRECTOTCORR
0,40.0,-55.625,2014,1,3.46
1,40.0,-55.000,2014,1,3.87
2,40.0,-54.375,2014,1,4.73
3,40.0,-53.750,2014,1,5.51
4,40.0,-53.125,2014,1,6.42


### 2.6. Combine Weather Variables

Combine temperature, humidity, wind speed, and precipitation using geographic coordinates and observation dates.

Save the combined regional datasets in Parquet format to avoid repeated processing.

In [18]:
# Combine weather variables by region and year
output_dir = Path("data/power_processed")
output_dir.mkdir(exist_ok=True)
RUN_POWER_PROCESSING = False

keys = ["LAT", "LON", "YEAR", "DOY"]

for region in (regions.itertuples(index=False) if RUN_POWER_PROCESSING else []):
    lat, lon = int(region.region_lat), int(region.region_lon)

    for year in years:
        output = output_dir / f"weather_{lat}_{lon}_{year}.parquet"

        if output.exists():
            continue

        weather_data = None

        for variable in weather_variables:
            file = power_dir / f"POWER_{lat}_{lon}_{year}_{variable}.csv"
            df = pd.read_csv(file, skiprows=9, na_values=-999)

            weather_data = df if weather_data is None else weather_data.merge(
                df, on=keys, how="outer", validate="one_to_one"
            )

        weather_data.to_parquet(output, index=False)

print("Processed files:", len(list(output_dir.glob("*.parquet"))))

Processed files: 624


### 2.7. Validate Processed Weather Data

Verify the structure and completeness of the combined NASA POWER weather data before integration with FIRMS.

In [19]:
# Validate processed weather data
processed_files = list(output_dir.glob("*.parquet"))

power_sample = pd.read_parquet(processed_files[0])

print("Processed files:", len(processed_files))
print("Columns:", power_sample.columns.tolist())
print("Missing values:")
print(power_sample.isna().sum())

display(power_sample.head())

Processed files: 624
Columns: ['LAT', 'LON', 'YEAR', 'DOY', 'T2M', 'RH2M', 'WS10M', 'PRECTOTCORR']
Missing values:
LAT            0
LON            0
YEAR           0
DOY            0
T2M            0
RH2M           0
WS10M          0
PRECTOTCORR    0
dtype: int64


,LAT,LON,YEAR,DOY,T2M,RH2M,WS10M,PRECTOTCORR
0,40.0,-55.625,2014,1,12.21,81.76,13.40,3.46
1,40.0,-55.625,2014,2,11.19,79.14,7.64,2.57
2,40.0,-55.625,2014,3,18.22,80.17,19.06,23.99
3,40.0,-55.625,2014,4,8.79,80.26,16.25,3.34
4,40.0,-55.625,2014,5,12.48,77.71,7.65,0.46


### 2.8. Prepare FIRMS for Weather Integration

Assign each FIRMS observation to its corresponding NASA POWER download region.

Weather observations will then be matched by date and nearest geographic location, accounting for differences between FIRMS and NASA POWER spatial grids.


In [20]:
# Assign download regions to FIRMS observations
firms_daily["region_lat"] = np.floor((firms_daily["grid_lat"] + 0.25) / 8) * 8
firms_daily["region_lon"] = np.floor((firms_daily["grid_lon"] + 0.25) / 8) * 8

print("Observations:", len(firms_daily))
print("Regions:", firms_daily[["region_lat", "region_lon"]].drop_duplicates().shape[0])

display(firms_daily.head())

Observations: 203744
Regions: 48


,grid_lat,grid_lon,acq_date,fire_count,max_frp,mean_frp,total_frp,region_lat,region_lon
0,41.5,-83.0,2019-04-06,1,3.19,3.190000,3.19,40.0,-88.0
1,41.5,-83.0,2019-06-22,3,18.05,10.963333,32.89,40.0,-88.0
2,41.5,-83.0,2019-12-17,1,12.75,12.750000,12.75,40.0,-88.0
3,41.5,-83.0,2021-03-14,2,0.63,0.600000,1.20,40.0,-88.0
4,41.5,-83.0,2024-02-07,2,4.67,4.220000,8.44,40.0,-88.0


### 2.9. Merge FIRMS and NASA POWER

Match each FIRMS observation with daily weather data using the nearest NASA POWER grid location and the observation date.

Process the data by region and year to reduce memory usage, and save the merged results in Parquet format.


In [21]:
from scipy.spatial import cKDTree

merged_dir = Path("data/firms_power")
merged_dir.mkdir(exist_ok=True)
RUN_POWER_MERGE = False

fire_groups = firms_daily.assign(
    year=firms_daily["acq_date"].dt.year
).groupby(["region_lat", "region_lon", "year"])

for (lat, lon, year), fire in (fire_groups if RUN_POWER_MERGE else []):
    output = merged_dir / f"merged_{int(lat)}_{int(lon)}_{year}.parquet"

    if output.exists():
        continue

    weather = pd.read_parquet(
        output_dir / f"weather_{int(lat)}_{int(lon)}_{year}.parquet"
    )

    weather["acq_date"] = pd.to_datetime(
        weather["YEAR"] * 1000 + weather["DOY"], format="%Y%j"
    )

    # Match each FIRMS cell to its nearest POWER location
    locations = weather[["LAT", "LON"]].drop_duplicates().reset_index(drop=True)
    cells = fire[["grid_lat", "grid_lon"]].drop_duplicates().copy()

    scale = np.cos(np.radians(float(lat) + 4))
    tree = cKDTree(locations[["LAT", "LON"]].values * [1, scale])

    nearest = tree.query(
        (cells[["grid_lat", "grid_lon"]].values + 0.25) * [1, scale]
    )[1]

    cells[["LAT", "LON"]] = locations.iloc[nearest].values

    merged = fire.merge(cells, on=["grid_lat", "grid_lon"])
    merged = merged.merge(
        weather.drop(columns=["YEAR", "DOY"]),
        on=["LAT", "LON", "acq_date"],
        how="left",
        validate="many_to_one"
    )

    merged.to_parquet(output, index=False)

print("Merged files:", len(list(merged_dir.glob("merged_*.parquet"))))
print("Expected groups:", fire_groups.ngroups)

Merged files: 468
Expected groups: 468


## 3. CWFIS: Fire Weather Indices

### 3.1. Load Historical CWFIS Data

Load Canadian Fire Weather Index observations from the 2010–2019 and 2020–2026 archives.

Keep observations from 2014 onward for integration with the FIRMS and NASA POWER datasets.


In [22]:
# Locate actual CWFIS CSV files
cwfis_files = sorted(
    file for file in Path("data").rglob("*.csv")
    if file.is_file() and file.name.startswith("cwfis_fwi")
)

print("CWFIS files:", len(cwfis_files))

for file in cwfis_files:
    print(file)

CWFIS files: 2
data\cwfis_fwi2010sv3.0_ll.csv\cwfis_fwi2010sv3.0_ll (1).csv
data\cwfis_fwi2020sv3.0_ll.csv\cwfis_fwi2020sv3.0_ll (1).csv


### 3.2. Load and Filter CWFIS Data

Combine the historical CWFIS datasets and retain observations from 2014 onward.

Keep station coordinates, observation dates, and four fire weather indices (FFMC, ISI, BUI, FWI).


In [23]:
# Load cached CWFIS results or original data
cwfis_path = Path("data/firms_cwfis.parquet")
USE_CACHED_CWFIS = cwfis_path.exists()

if USE_CACHED_CWFIS:
    firms_cwfis = pd.read_parquet(cwfis_path)
    print("Loaded processed CWFIS data:", firms_cwfis.shape)
else:
    columns = ["rep_date", "lat", "lon", "ffmc", "isi", "bui", "fwi"]

    cwfis = pd.concat(
        [pd.read_csv(file, usecols=columns) for file in cwfis_files],
        ignore_index=True
    )

    cwfis["rep_date"] = pd.to_datetime(cwfis["rep_date"])
    cwfis = cwfis[cwfis["rep_date"] >= "2014-01-01"].copy()

    print("Raw CWFIS data:", cwfis.shape)


Loaded processed CWFIS data: (203744, 14)


### 3.3. Prepare CWFIS Data for Integration

Keep observations with complete fire weather indices and prepare daily dates for spatial and temporal matching with FIRMS.

CWFIS data is currently available through September 2026.


In [24]:
# Prepare CWFIS data only when needed
if not USE_CACHED_CWFIS:
    indices = ["ffmc", "isi", "bui", "fwi"]

    cwfis_clean = cwfis.dropna(subset=indices).copy()
    cwfis_clean["date"] = cwfis_clean["rep_date"].dt.normalize()

    print("Available observations:", len(cwfis_clean))
    print("Unique station locations:",
          cwfis_clean[["lat", "lon"]].drop_duplicates().shape[0])
else:
    print("CWFIS preparation skipped: cached data available.")

CWFIS preparation skipped: cached data available.


### 3.4. Merge FIRMS and CWFIS

Integrate daily FFMC, ISI, BUI, and FWI values from the nearest CWFIS station.

Preserve all FIRMS observations and retain missing indices when station data is unavailable. Station distance is recorded to assess geographic matching quality.


In [25]:
# Load or calculate FIRMS + CWFIS integration
from sklearn.neighbors import BallTree

if USE_CACHED_CWFIS:
    print("CWFIS integration loaded from cache.")

else:
    station_daily = cwfis_clean.groupby(
        ["date", "lat", "lon"], as_index=False
    )[["ffmc", "isi", "bui", "fwi"]].mean()

    station_groups = station_daily.groupby("date")
    results = []

    for date, fire in firms_daily.groupby("acq_date"):
        fire = fire.copy()
        fire[["ffmc", "isi", "bui", "fwi", "distance_km"]] = np.nan

        if date in station_groups.groups:
            stations_day = station_groups.get_group(date)

            tree = BallTree(
                np.radians(stations_day[["lat", "lon"]]),
                metric="haversine"
            )

            points = fire[["grid_lat", "grid_lon"]].to_numpy() + 0.25
            distances, nearest = tree.query(np.radians(points), k=1)

            fire[["ffmc", "isi", "bui", "fwi"]] = (
                stations_day.iloc[nearest[:, 0]][
                    ["ffmc", "isi", "bui", "fwi"]
                ].to_numpy()
            )
            fire["distance_km"] = distances[:, 0] * 6371

        results.append(fire)

    firms_cwfis = pd.concat(results, ignore_index=True)

    # Exclude station matches beyond 100 km
    indices = ["ffmc", "isi", "bui", "fwi"]

    firms_cwfis.loc[
        firms_cwfis["distance_km"] > 100, indices
    ] = np.nan

    firms_cwfis.to_parquet(cwfis_path, index=False)

print("Observations:", len(firms_cwfis))
print("Missing FWI:", firms_cwfis["fwi"].isna().sum())

CWFIS integration loaded from cache.
Observations: 203744
Missing FWI: 42502


### 3.5. Validate CWFIS Matching

Verify the geographic matching quality after applying the 100 km distance limit.

Ensure that no fire observation receives weather indices from a station located more than 100 km away.


In [26]:
# Check CWFIS matching quality after the 100 km limit
print("Total observations:", len(firms_cwfis))
print("FWI available:", firms_cwfis["fwi"].notna().sum())
print("FWI missing:", firms_cwfis["fwi"].isna().sum())

print("Matches within 50 km:",
      (firms_cwfis["distance_km"] <= 50).sum())
print("Matches between 50 and 100 km:",
      ((firms_cwfis["distance_km"] > 50) &
       (firms_cwfis["distance_km"] <= 100)).sum())
print("Matches beyond 100 km with FWI:",
      ((firms_cwfis["distance_km"] > 100) &
       firms_cwfis["fwi"].notna()).sum())

Total observations: 203744
FWI available: 161242
FWI missing: 42502
Matches within 50 km: 112651
Matches between 50 and 100 km: 48591
Matches beyond 100 km with FWI: 0


## 4. MRDEM: Topographic Features

### 4.1. Load the Elevation Dataset

Use the Canadian Digital Elevation Model (MRDEM) to extract terrain elevation.

Elevation is extracted at the center of each unique FIRMS grid cell using a 30 m resolution raster.

Local slope extraction was deferred due to performance issues with the remote raster.


In [27]:
# MRDEM dataset
dem_path = Path("data/mrdem-30-dtm.vrt")

if Path("data/grid_elevation.parquet").exists():
    print("MRDEM elevation already extracted.")
else:
    with rasterio.open(dem_path) as dem:
        print("Coordinate system:", dem.crs)
        print("Resolution:", dem.res)
        print("NoData value:", dem.nodata)

MRDEM elevation already extracted.


### 4.2. Extract Elevation for All Grid Cells

Extract terrain elevation at the center of each FIRMS grid cell using the 30 m MRDEM raster.

Save the results in Parquet format to avoid repeated access to the remote raster.

If the processed file already exists, load it directly instead of repeating the extraction.

In [28]:
# Load saved elevations or extract them if needed
elevation_path = Path("data/grid_elevation.parquet")

if elevation_path.exists():
    grid_cells = pd.read_parquet(elevation_path)
else:
    with rasterio.open(dem_path) as dem:
        x, y = rasterio.warp.transform(
            "EPSG:4326",
            dem.crs,
            grid_cells["longitude"].tolist(),
            grid_cells["latitude"].tolist()
        )

        elevations = [
            value[0] for value in dem.sample(zip(x, y), masked=True)
        ]

    grid_cells["elevation"] = np.array(elevations, dtype=float)
    grid_cells.to_parquet(elevation_path, index=False)

print("Grid cells:", len(grid_cells))
print("Elevation available:", grid_cells["elevation"].notna().sum())
print("Elevation missing:", grid_cells["elevation"].isna().sum())

Grid cells: 3977
Elevation available: 3975
Elevation missing: 2


### 4.3. Topographic Features: Finalization

Elevation was successfully extracted from MRDEM at 30 m resolution.

Local slope extraction was deferred because repeated reads from the remote raster caused performance and access issues.

The pipeline retains elevation as a topographic feature. Slope may be added later if needed.


In [29]:
# Finalize topographic features
grid_cells["slope"] = np.nan

grid_cells.to_parquet("data/grid_topography.parquet", index=False)

print("Grid cells:", len(grid_cells))
print("Elevation available:", grid_cells["elevation"].notna().sum())
print("Slope calculated: No (deferred)")

Grid cells: 3977
Elevation available: 3975
Slope calculated: No (deferred)


## 5. FBP: Fuel Types

### 5.1. Load the Fuel Type Raster

Use the Canadian Fire Behaviour Prediction (FBP) fuel type map to identify the fuel type at the center of each FIRMS grid cell.

The 2014 fuel map is used as a static geographic feature for the study period.


In [30]:
# Locate the FBP fuel type raster
fbp_files = sorted(
    Path("data/National_FBP_Fueltypes_version2014b").rglob("*.tif")
)

print("FBP raster files:", len(fbp_files))

for file in fbp_files:
    print(file)

FBP raster files: 1
data\National_FBP_Fueltypes_version2014b\nat_fbpfuels_2014b.tif


### 5.2. Inspect the Fuel Type Raster

Inspect the FBP raster's coordinate system, spatial resolution, and NoData value before extracting fuel type codes.

Fuel types will be assigned using the center of each FIRMS grid cell.

In [31]:
# Inspect FBP raster only if extraction is needed
fuel_path = Path("data/grid_fueltypes.parquet")

if fuel_path.exists():
    print("FBP fuel types already extracted.")
else:
    fbp_path = fbp_files[0]

    with rasterio.open(fbp_path) as fbp:
        print("Coordinate system:", fbp.crs)
        print("Resolution:", fbp.res)
        print("Dimensions:", fbp.width, "x", fbp.height)
        print("NoData value:", fbp.nodata)
        print("Data type:", fbp.dtypes[0])

FBP fuel types already extracted.


### 5.3. Extract Fuel Types

Extract FBP fuel type codes at the center of each unique FIRMS grid cell using the 250 m resolution raster.

Save the results in Parquet format. If the processed file already exists, load it directly to avoid repeating the extraction.

The 2014 fuel type map is treated as a static geographic feature throughout the study period.

In [32]:
# Load saved fuel types or extract them if needed
fuel_path = Path("data/grid_fueltypes.parquet")

if fuel_path.exists():
    grid_cells = pd.read_parquet(fuel_path)
else:
    with rasterio.open(fbp_path) as fbp:
        x_fbp, y_fbp = rasterio.warp.transform(
            "EPSG:4326",
            fbp.crs,
            grid_cells["longitude"].tolist(),
            grid_cells["latitude"].tolist()
        )

        fuel_values = list(
            fbp.sample(zip(x_fbp, y_fbp), masked=True)
        )

    grid_cells["fuel_type"] = np.ma.filled(
        np.ma.array([v[0] for v in fuel_values]),
        np.nan
    )

    grid_cells.to_parquet(fuel_path, index=False)

print("Grid cells:", len(grid_cells))
print("Fuel types available:", grid_cells["fuel_type"].notna().sum())
print("Fuel types missing:", grid_cells["fuel_type"].isna().sum())

Grid cells: 3977
Fuel types available: 3782
Fuel types missing: 195


### 5.4. Fuel Type Distribution

Examine the distribution of FBP fuel type codes across FIRMS grid cells.

Keep the original codes until their meanings are verified using the official FBP legend.

In [33]:
# Distribution of FBP fuel type codes
print(grid_cells["fuel_type"].value_counts(dropna=False).sort_index())

fuel_type
101.0    421
102.0    898
103.0     17
105.0     53
107.0      1
108.0    534
109.0    544
116.0    159
118.0    403
119.0    128
120.0     36
121.0      8
122.0    580
NaN      195
Name: count, dtype: int64


## 6. Final Dataset Integration

### 6.1. Merge All Data Sources

Combine NASA FIRMS, NASA POWER, CWFIS, MRDEM, and FBP into a single machine learning dataset.

Use the processed Parquet files to avoid repeating data extraction and preprocessing.

Preserve all FIRMS observations, with daily maximum FRP as the prediction target.

In [34]:
# Load processed FIRMS + POWER files
merged_files = sorted(Path("data/firms_power").glob("merged_*.parquet"))

df = pd.concat(
    [pd.read_parquet(file) for file in merged_files],
    ignore_index=True
)

# Add CWFIS indices
cwfis_data = pd.read_parquet("data/firms_cwfis.parquet")

df = df.merge(
    cwfis_data[
        ["grid_lat", "grid_lon", "acq_date",
         "ffmc", "isi", "bui", "fwi", "distance_km"]
    ],
    on=["grid_lat", "grid_lon", "acq_date"],
    how="left",
    validate="one_to_one"
)

# Add topography and fuel types
topography = pd.read_parquet("data/grid_fueltypes.parquet")

df = df.merge(
    topography[
        ["grid_lat", "grid_lon", "elevation", "fuel_type"]
    ],
    on=["grid_lat", "grid_lon"],
    how="left",
    validate="many_to_one"
)

print("Final dataset shape:", df.shape)
print("Unique grid cells:",
      df[["grid_lat", "grid_lon"]].drop_duplicates().shape[0])
print("Date range:", df["acq_date"].min(), "to", df["acq_date"].max())
display(df.head())

Final dataset shape: (203744, 23)
Unique grid cells: 3977
Date range: 2014-01-01 00:00:00 to 2026-07-31 00:00:00


,grid_lat,grid_lon,acq_date,fire_count,max_frp,mean_frp,total_frp,region_lat,region_lon,year,...,RH2M,WS10M,PRECTOTCORR,ffmc,isi,bui,fwi,distance_km,elevation,fuel_type
0,47.0,-56.0,2014-09-25,1,0.90,0.90,0.90,40.0,-56.0,2014,...,76.80,5.68,0.00,NaN,NaN,NaN,NaN,132.140462,0.078980,NaN
1,47.0,-55.5,2014-07-21,1,0.97,0.97,0.97,40.0,-56.0,2014,...,85.36,2.95,0.47,82.4,3.1,14.3,4.1,94.467180,71.125893,108.0
2,47.0,-54.0,2014-10-27,1,2.45,2.45,2.45,40.0,-56.0,2014,...,94.12,3.76,8.22,21.8,0.0,0.4,0.0,19.663255,113.848206,101.0
3,47.0,-53.5,2014-07-21,1,8.64,8.64,8.64,40.0,-56.0,2014,...,78.88,2.68,0.21,92.4,13.8,48.9,27.5,35.854287,91.291985,118.0
4,47.0,-53.0,2014-05-17,1,2.15,2.15,2.15,40.0,-56.0,2014,...,80.68,3.29,0.00,86.5,6.0,7.9,5.7,29.792822,-0.035409,NaN


## 7. Final Data Quality Checks and Export

### 7.1. Check Data Quality

Verify the final dataset's completeness, uniqueness, and date coverage before exporting it for machine learning.

Identify missing values and confirm that the original FIRMS observations have been preserved.


In [35]:
# Final data quality checks
print("Total observations:", len(df))
print("Duplicate grid-days:",
      df.duplicated(["grid_lat", "grid_lon", "acq_date"]).sum())

print("\nMissing values (%):")
print((df.isna().mean() * 100).round(1).sort_values(ascending=False))

print("\nObservations by year:")
print(df["acq_date"].dt.year.value_counts().sort_index())

print("\nOriginal FIRMS last date:", firms_daily["acq_date"].max())
print("Final dataset last date:", df["acq_date"].max())

Total observations: 203744
Duplicate grid-days: 0

Missing values (%):
fwi            20.9
bui            20.9
isi            20.9
ffmc           20.9
fuel_type       1.0
T2M             0.0
elevation       0.0
distance_km     0.0
PRECTOTCORR     0.0
WS10M           0.0
RH2M            0.0
grid_lat        0.0
grid_lon        0.0
LAT             0.0
year            0.0
region_lon      0.0
region_lat      0.0
total_frp       0.0
mean_frp        0.0
max_frp         0.0
fire_count      0.0
acq_date        0.0
LON             0.0
dtype: float64

Observations by year:
acq_date
2014    17125
2015    18300
2016    11939
2017    18479
2018    14865
2019    10917
2020     9794
2021    17391
2022    11732
2023    30057
2024    16618
2025    19113
2026     7414
Name: count, dtype: int64

Original FIRMS last date: 2026-07-31 00:00:00
Final dataset last date: 2026-07-31 00:00:00


### 7.2. FIRMS Data Coverage

The FIRMS dataset contains two files:

- **Historical Archive:** Includes the `type` column and covers observations through July 31, 2026.
- **Near Real-Time (NRT):** Covers August 1 to November 2, 2026, but does not contain the `type` column.

Historical observations were filtered using `type == 0` to retain vegetation fires.

NRT observations were excluded because the same filtering criterion could not be applied.

**Final decision:** The dataset covers January 2014 to July 2026, ensuring consistent preprocessing across the study period.


### 7.3. Final Dataset Export

Prepare and export the final machine learning dataset.

Retain the prediction target, geographic and temporal information, and environmental features from NASA POWER, CWFIS, MRDEM, and FBP.

Exclude intermediate processing columns and save the dataset in Parquet format.

In [36]:
# Select final dataset columns
final_columns = [
    "grid_lat", "grid_lon", "acq_date",
    "fire_count", "max_frp",
    "T2M", "RH2M", "WS10M", "PRECTOTCORR",
    "ffmc", "isi", "bui", "fwi",
    "distance_km", "elevation", "fuel_type"
]

final_df = df[final_columns].copy()

# Save final ML dataset
final_df.to_parquet("data/wildfire_ml_dataset.parquet", index=False)

print("Final dataset shape:", final_df.shape)
print("Missing values (%):")
print((final_df.isna().mean() * 100).round(1))
print("Saved: data/wildfire_ml_dataset.parquet")

Final dataset shape: (203744, 16)
Missing values (%):
grid_lat        0.0
grid_lon        0.0
acq_date        0.0
fire_count      0.0
max_frp         0.0
T2M             0.0
RH2M            0.0
WS10M           0.0
PRECTOTCORR     0.0
ffmc           20.9
isi            20.9
bui            20.9
fwi            20.9
distance_km     0.0
elevation       0.0
fuel_type       1.0
dtype: float64
Saved: data/wildfire_ml_dataset.parquet


### 7.4. Dataset Limitations

The final dataset contains 203,744 daily fire observations from January 2014 to July 2026.

**Main limitations:**

- **CWFIS:** 20.9% of observations have missing fire weather indices because no suitable station was available within 100 km.
- **MRDEM:** Elevation was extracted at grid cell centers. Local terrain slope was not included.
- **FBP:** The 2014 fuel type map was used as a static feature throughout the study period.
- **FIRMS:** Near Real-Time observations were excluded to maintain consistent vegetation fire filtering.
- **Prediction target:** `max_frp` represents the maximum detected fire radiative power (MW) for each grid cell and day. It does not represent future wildfire occurrence or intensity.

Missing values were preserved for the machine learning preprocessing stage.
